# FEWS Agent Tools Validation

This notebook validates the tools available in the FEWS Configuration Agent without using the LLM. It directly tests each tool's functionality to ensure they work as expected.

In [27]:
import sys
from pathlib import Path
from IPython.display import display, Markdown

# Add parent directory to path
sys.path.append(Path().absolute().parent.as_posix())

# Import required modules
from talkwithfews.tools import (
    execute_query,
    explain_query,
    search_workflows,
    get_workflow_details,
    search_timeseries,
    get_timeseries_details,
    search_transformations,
    get_transformation_details,
    analyze_upstream_dependencies,
    analyze_downstream_dependencies,
    find_workflow_call_hierarchy,
    get_database_schema,
    analyze_dead_points
)

from talkwithfews.models import (
    Deps,
    SQLQueryInput,
    WorkflowSearchInput,
    TimeSeriesSearchInput,
    TransformationSearchInput,
    DependencyAnalysisInput
)

from talkwithfews.utils import connect_to_database
from talkwithfews.logging_utils import setup_logging
from pydantic_ai import RunContext

# Set logging level to ERROR at the start of the notebook
setup_logging(log_level="ERROR")

In [28]:
# Set up database connection
db_path = '../db/fews_config.db3'
db_connection = connect_to_database(db_path)

# Create dependencies object
deps = Deps(
    db_path=db_path, 
    db_connection=db_connection
)

# Create run context with mock values for required parameters
ctx = RunContext(
    deps=deps,
    model="mock-model",  # Mock model name since we're not using LLM
    usage={              # Mock usage statistics
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "total_tokens": 0
    },
    prompt="mock-prompt" # Mock prompt since we're not using LLM
)

## 1. Database Schema Validation

In [29]:
# Test get_database_schema
schema = await get_database_schema(ctx)
print("Database Schema:")
print(f"Number of tables: {len(schema['tables'])}")
print(f"Number of relationships: {len(schema['relationships'])}")

print("\nRaw Schema Structure:")
print("Tables:")
for table_name, columns in schema['tables'].items():
    print(f"- {table_name}:")
    print(f"  Columns: {', '.join(columns)}")

print("\nRelationships:")
for rel in schema['relationships']:
    print(f"{rel}")

print("\nSample Data:")
for table_name, samples in schema['sample_data'].items():
    print(f"\n{table_name}:")
    for sample in samples[:2]:  # Show first 2 samples for brevity
        print(f"  {sample}")
        
print("\nTables:")
for table_name, columns in schema['tables'].items():
    print(f"- {table_name}: {len(columns)} columns")

Database Schema:
Number of tables: 7
Number of relationships: 8

Raw Schema Structure:
Tables:
- call_stacks:
  Columns: id, wf_id, desc, wffn, parent_id, props_set_id
- endpoints:
  Columns: tss_id, desc
- links:
  Columns: trafo_id, tss_id, inout
- modins_calls:
  Columns: id, call_stack_id, modins, props_set_id
- props_sets:
  Columns: id, key, value
- trafos:
  Columns: id, mic_id, ord_nr, desc
- ts_sets:
  Columns: id, modins, vtype, parid, qualifiers, tstype, timess, mic_id

Relationships:
- call_stacks.parent_id → call_stacks.id
- call_stacks.props_set_id → props_sets.id
- endpoints.tss_id → ts_sets.id
- links.tss_id → ts_sets.id
- links.trafo_id → trafos.id
- modins_calls.call_stack_id → call_stacks.id
- modins_calls.props_set_id → props_sets.id
- trafos.mic_id → modins_calls.id

Sample Data:

call_stacks:
  {'id': 1, 'wf_id': 'Import_Waterwatch', 'desc': 'Importeer waterwatch en verwerk naar peilgebieden', 'wffn': 'Import_Waterwatch', 'parent_id': None, 'props_set_id': None}
 

## 2. Query Execution Validation

In [30]:
# Test execute_query with a simple query
simple_query = "SELECT COUNT(*) as count FROM trafos"
result = await execute_query(ctx, SQLQueryInput(query=simple_query))
print("Number of transformations:")
print(result.results[0]['count'])

# Test explain_query
explanation = await explain_query(ctx, SQLQueryInput(query=simple_query))
print("\nQuery explanation:")
print(explanation)

Number of transformations:
2387

Query explanation:
This query: SELECT COUNT(*) as count FROM trafos

Is designed to retrieve data from the FEWS configuration database.


## 3. Workflow Tools Validation

In [31]:
# Test search_workflows
workflow_search = await search_workflows(ctx, WorkflowSearchInput(name_pattern="water"))
print("Workflows containing 'water':")
for workflow in workflow_search.results:
    print(f"- {workflow['wf_id']} (ID: {workflow['id']}) - {workflow['desc']}")

# Test get_workflow_details if we found any workflows
if workflow_search.results:
    workflow_id = workflow_search.results[0]['id']
    details = await get_workflow_details(ctx, {'workflow_id': workflow_id})
    print("\nFirst workflow details:")
    print(f"ID: {details.results[0]['id']}")
    print(f"Workflow ID: {details.results[0]['wf_id']}")
    print(f"Description: {details.results[0]['desc']}")

Workflows containing 'water':
- Export_rijkswaterstaat (ID: 15) - Exporteer bestanden voor FEWS omgeving van RWS
- Gebiedsgemiddelden (ID: 41) - Berekend de gemiddelde waterstand uit alle loggers binnen 1 praktijkgebied
- Import_Waterwatch (ID: 1) - Importeer waterwatch en verwerk naar peilgebieden
- ProcesOverzicht (ID: 32) - Genereren van de watervraag voor heel Rijnland
- Waterbalans_boezem (ID: 46) - Bereken waterbalans van de boezem
- Waterbalans_polder (ID: 48) - Bereken waterbalans van polders
- Waterbalans_praktijkpeilgebied (ID: 47) - Bereken waterbalans van praktijkpeilgebieden

First workflow details:
ID: 15
Workflow ID: Export_rijkswaterstaat
Description: Exporteer bestanden voor FEWS omgeving van RWS


## 4. Time Series Tools Validation

In [32]:
# Test search_timeseries
ts_search = await search_timeseries(ctx, TimeSeriesSearchInput(
    parameter_id="water",  # Search in parameter IDs
    qualifier_pattern=None,
    module_instance=None
))
print("Time series containing 'water':")
for ts in ts_search.results:
    print(f"- {ts['parid']} (ID: {ts['id']}) - Qualifiers: {ts['qualifiers']}")

# Test get_timeseries_details if we found any time series
if ts_search.results:
    ts_id = ts_search.results[0]['id']
    details = await get_timeseries_details(ctx, {'timeseries_id': ts_id})
    print("\nFirst time series details:")
    print(f"Parameter ID: {details.results[0]['parid']}")
    print(f"Qualifiers: {details.results[0]['qualifiers']}")
    print(f"Type: {details.results[0]['tstype']}")

Time series containing 'water':
- water.T.meting (ID: 104) - Qualifiers: 
- waterberging.aanvoer (ID: 339) - Qualifiers: 
- waterberging.afvoer (ID: 343) - Qualifiers: 
- waterbezwaar (ID: 353) - Qualifiers: 
- waterberging (ID: 354) - Qualifiers: 
- waterstand_verschil (ID: 450) - Qualifiers: 
- watervraag (ID: 460) - Qualifiers: 

First time series details:
Parameter ID: water.T.meting
Qualifiers: 
Type: external historical


## 5. Transformation Tools Validation

In [33]:
# Test search_transformations
trans_search = await search_transformations(ctx, TransformationSearchInput(description_pattern="water"))
print("Transformations containing 'water':")
for trans in trans_search.results:
    print(f"- {trans['desc']} (ID: {trans['id']})")

# Test get_transformation_details if we found any transformations
if trans_search.results:
    trans_id = trans_search.results[0]['id']
    details = await get_transformation_details(ctx, {'transformation_id': trans_id})
    print("\nFirst transformation details:")
    print(f"Description: {details.results[0]['desc']}")
    print(f"Module Instance: {details.results[0]['modins']}")

Transformations containing 'water':
- E.openwater (ID: 7)
- E.volume.openwater (ID: 8)
- Waterberging.aanvoer (ID: 44)
- Waterberging.afvoer (ID: 45)
- sommenreeks_wateraanvoer_poldervierambacht (ID: 73)
- gem waterstand per prkgeb (ID: 563)
- Export_rijkswaterstaat (ID: 841)
- Export_rijkswaterstaat (ID: 842)
- Export_rijkswaterstaat (ID: 843)
- gem waterstand per prkgeb (ID: 855)
- Waterstand verschil (ID: 871)
- waterberging (ID: 872)
- Watervraag (ID: 876)
- Waterbezwaar (ID: 877)
- Waterstand verschil (ID: 891)
- waterberging (ID: 892)
- Watervraag (ID: 896)
- Waterbezwaar (ID: 897)
- Merge_waterstands_verschil (ID: 908)
- Merge_waterstands_verschil (ID: 919)
- ImportWIWB_WATERSETUPEPS-6u (ID: 1193)
- ImportWIWB_WATERSETUPEPS-6u (ID: 1194)
- ImportWaterwatch (ID: 1197)
- ImportWaterwatch (ID: 1198)
- ImportWaterwatch (ID: 1199)
- ImportWaterwatch (ID: 1200)
- inlaatstand_combi.Leeghwater (ID: 1323)
- Q.inlaat.Leeghwater (ID: 1324)
- bereken afvoer Leeghwater (ID: 1325)
- bereken a

## 6. Dependency Analysis Tools Validation

In [34]:
# Test analyze_upstream_dependencies
if ts_search.results:
    ts_id = ts_search.results[0]['id']
    upstream = await analyze_upstream_dependencies(ctx, DependencyAnalysisInput(
        entity_id=ts_id,
        entity_type='timeseries'
    ))
    print("Upstream dependencies:")
    print(f"Number of producers: {len(upstream.results)}")
    if upstream.results:
        print("First producer:")
        print(f"- Type: {upstream.results[0]['type'] if 'type' in upstream.results[0] else 'N/A'}")
        print(f"- ID: {upstream.results[0]['id']}")

    # Test analyze_downstream_dependencies
    downstream = await analyze_downstream_dependencies(ctx, DependencyAnalysisInput(
        entity_id=ts_id,
        entity_type='timeseries'
    ))
    print("\nDownstream dependencies:")
    print(f"Number of consumers: {len(downstream.results)}")
    if downstream.results:
        print("First consumer:")
        print(f"- Type: {downstream.results[0]['type'] if 'type' in downstream.results[0] else 'N/A'}")
        print(f"- ID: {downstream.results[0]['id']}")

Upstream dependencies:
Number of producers: 0

Downstream dependencies:
Number of consumers: 0


## 7. Workflow Call Hierarchy Validation

In [35]:
# Test find_workflow_call_hierarchy
if workflow_search.results:
    root_workflow_id = workflow_search.results[0]['id']
    hierarchy = await find_workflow_call_hierarchy(ctx, {'root_workflow_id': root_workflow_id})  # Changed key name
    print("Workflow call hierarchy:")
    print(f"Number of levels: {len(hierarchy.results)}")
    
    # Group results by level
    levels = {}
    for item in hierarchy.results:
        level = item['level']
        if level not in levels:
            levels[level] = []
        levels[level].append(item)
    
    # Print each level
    for level, workflows in sorted(levels.items()):
        print(f"\nLevel {level}:")
        print(f"Number of workflows: {len(workflows)}")
        if workflows:
            print(f"First workflow: {workflows[0]['wf_id']}")

Workflow call hierarchy:
Number of levels: 1

Level 0:
Number of workflows: 1
First workflow: Export_rijkswaterstaat


## 8. Dead Points Analysis Validation

In [36]:
# Test analyze_dead_points
dead_points = await analyze_dead_points(ctx)
print("Dead points analysis:")
print(f"Total dead points found: {dead_points['count']}")
if dead_points['dead_points']:
    print("\nFirst dead point:")
    first_dp = dead_points['dead_points'][0]
    print(f"- ID: {first_dp['id']}")
    print(f"- Parameter ID: {first_dp['parid']}")
    print(f"- Module Instance: {first_dp['modins']}")

Dead points analysis:
Total dead points found: 79

First dead point:
- ID: 735
- Parameter ID: ProcentBeschikbaar
- Module Instance: $MODULE_INSTANCE_ID$


## Cleanup

In [37]:
# Close database connection
db_connection.close()